In [ ]:
%run setup.py

# 05 · Aprendizaje por refuerzo — Multi-Armed Bandits

## Definición del entorno

| Elemento | Definición |
| --- | --- |
| **Estado** | El problema es un bandit sin estado (*stateless*): en cada episodio el agente observa el mismo estado (un comprador potencial). |
| **Acciones** | Publicar con precio **Alto**, **Medio** o **Bajo**. |
| **Recompensa** | Si el comprador acepta, el agente recibe el precio de venta; si rechaza, recibe 0. Así se premia vender, pero también vender caro. |
| **Terminación** | Cada episodio termina tras una interacción con un comprador; el entrenamiento termina a los 1.000 episodios. |
| **Parámetros** | `epsilon = 0.1` (10% exploración), 3 acciones, 1.000 episodios, `random_state`. |

El agente usa **ε-greedy**: explora al azar con probabilidad ε y explota la
acción con mayor recompensa promedio estimada.

## 5.1 Fundamento teórico

En un **Multi-Armed Bandit** el agente elige repetidamente entre `K` acciones
con recompensa esperada desconocida `μa`. Para cada acción estima su valor:

`Q(a) = (suma de recompensas obtenidas con a) / (veces que eligió a)`

La política **ε-greedy** elige la acción con mayor `Q(a)` con probabilidad
`1 − ε` y una acción al azar con probabilidad `ε`. Así equilibra
**explotación** (usar lo aprendido) y **exploración** (probar alternativas).
El objetivo es maximizar la recompensa acumulada y reducir el arrepentimiento
(*regret*) frente a la mejor acción posible.

In [ ]:
# CELDA 5: Definición del entorno simulado
random.seed(42)

precios = {"Alto": 1_000_000, "Medio": 800_000, "Bajo": 500_000}
acciones = list(precios)
# Probabilidad oculta de que un comprador acepte cada precio
probabilidades_venta = {"Alto": 0.10, "Medio": 0.60, "Bajo": 0.85}

print("Precio esperado por acción (probabilidad x precio):")
for accion in acciones:
    esperado = probabilidades_venta[accion] * precios[accion]
    print(f"  {accion:>5}: ${esperado:,.0f}")

In [ ]:
# Entrenamiento ε-greedy
epsilon = 0.1
episodios = 1000
intentos = {a: 0 for a in acciones}
recompensas_totales = {a: 0 for a in acciones}
ventas = {a: 0 for a in acciones}
historial_recompensa = []
historial_acciones = []

for _ in range(episodios):
    if random.random() < epsilon:
        accion = random.choice(acciones)                      # explorar
    else:
        promedios = {
            a: (recompensas_totales[a] / intentos[a] if intentos[a] else 0)
            for a in acciones
        }
        accion = max(promedios, key=promedios.get)            # explotar

    intentos[accion] += 1
    if random.random() < probabilidades_venta[accion]:
        recompensa = precios[accion]                          # venta exitosa
        ventas[accion] += 1
    else:
        recompensa = 0                                        # venta fallida

    recompensas_totales[accion] += recompensa
    historial_recompensa.append(sum(recompensas_totales.values()))
    historial_acciones.append(accion)

In [ ]:
# Resultados por estrategia
resumen = pd.DataFrame({
    "accion": acciones,
    "intentos": [intentos[a] for a in acciones],
    "ventas": [ventas[a] for a in acciones],
    "recompensa_total": [recompensas_totales[a] for a in acciones],
    "recompensa_promedio": [
        recompensas_totales[a] / intentos[a] if intentos[a] else 0 for a in acciones
    ],
})
display(resumen.style.format({
    "recompensa_total": "${:,.0f}",
    "recompensa_promedio": "${:,.0f}",
}))

mejor = max(acciones, key=lambda a: recompensas_totales[a] / intentos[a] if intentos[a] else 0)
print(f"Mejor estrategia aprendida: {mejor}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(historial_recompensa)
axes[0].set_title("Recompensa acumulada del agente")
axes[0].set_xlabel("Episodio (comprador simulado)")
axes[0].set_ylabel("Ingresos acumulados (AUD)")

for accion in acciones:
    proporcion = pd.Series(historial_acciones).eq(accion).rolling(50, min_periods=1).mean()
    axes[1].plot(proporcion, label=accion)
axes[1].set_title("Evolución de la elección de acciones (media móvil 50)")
axes[1].set_xlabel("Episodio")
axes[1].set_ylabel("Proporción de elecciones")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Al inicio el agente explora las tres estrategias; con el tiempo concentra")
print("sus elecciones en la acción con mayor ingreso esperado.")